# HDBSCAN Finetuning — Original vs Pruned Clusters

Fine-tunes a MiniLM model on hard HDBSCAN cluster assignments using
`hardfinetuning_crossentropy.py` from the RecsysUpgrade repo.

Runs two comparisons back-to-back:
1. **Original HDBSCAN** — all non-noise clusters from `hdbscan_clusters.csv`
2. **Pruned HDBSCAN** — LSH-pruned clusters from `hdbscan_clusters_pruned.csv`

Cluster IDs are remapped to contiguous 0…N-1 integers before training
(noise points, label = −1, are excluded from both runs).

Paths and cluster files are produced by `UMAP_HDBSCAN.ipynb`.


### Step 1: Setup (drive mounting, paths)


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE          = "/content/drive/MyDrive/S2026/Spotify Playlist Data/Processed Data/calced_embeddings"
PROJECT_BASE        = "/content/drive/MyDrive/S2026/CS 274/Playlist-Recommender"
EMBEDDINGS_PKL      = f"{DRIVE_BASE}/mean_pooled_embeddings.pkl"
CLUSTERS_OUTPUT_DIR = f"{PROJECT_BASE}/umap clusters"

FINETUNED_MODEL_DIR         = f"{PROJECT_BASE}/hdbscan_finetuned_model"
FINETUNED_PRUNED_MODEL_DIR  = f"{PROJECT_BASE}/hdbscan_finetuned_model_pruned"
REPO_DIR                    = "/content/RecsysUpgrade"

os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
os.makedirs(FINETUNED_PRUNED_MODEL_DIR, exist_ok=True)
print("Paths configured.")


### Step 2: Clone repo and install requirements


In [ ]:
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/siddmohanty111/RecsysUpgrade.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)


In [ ]:
%pip install -r {REPO_DIR}/requirements.txt -q


### Step 3: Imports


In [ ]:
import importlib.util
import pickle

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


### Step 4: Load finetuning module and playlist titles


In [ ]:
ft_hard_path = os.path.join(
    REPO_DIR, "PlaylistRecsysUpgrade", "finetuning", "hardfinetuning_crossentropy.py"
)
spec    = importlib.util.spec_from_file_location("hardfinetuning_crossentropy", ft_hard_path)
hard_ft = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hard_ft)
print("hardfinetuning_crossentropy loaded.")

# Load playlist titles from the embeddings pkl (same source as UMAP_HDBSCAN.ipynb)
with open(EMBEDDINGS_PKL, "rb") as f:
    raw_data = pickle.load(f)

playlist_titles = (
    raw_data.get("playlist_titles", {})
    if isinstance(raw_data, dict)
    else {}
)
print(f"Playlist titles available: {len(playlist_titles)}")


---
## Part 1: Original HDBSCAN Clusters

Fine-tune on the full HDBSCAN cluster assignments (`hdbscan_clusters.csv`).
Noise points (cluster = −1) are removed. Cluster IDs are remapped to 0…N-1.


#### Load and prepare original HDBSCAN cluster dataset


In [ ]:
clusters_df = pd.read_csv(
    os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_clusters.csv"),
    dtype={"pid": str},
)
print(f"Loaded {len(clusters_df)} rows. Unique clusters (incl. noise): {clusters_df['cluster'].nunique()}")

# Remove noise points (HDBSCAN labels noise as -1)
clusters_df = clusters_df[clusters_df["cluster"] != -1].reset_index(drop=True)
print(f"After removing noise: {len(clusters_df)} playlists, {clusters_df['cluster'].nunique()} clusters")

# Remap cluster IDs to contiguous 0…N-1
unique_ids  = sorted(clusters_df["cluster"].unique())
id_map      = {old: new for new, old in enumerate(unique_ids)}
clusters_df["cluster"] = clusters_df["cluster"].map(id_map)
num_clusters_orig = clusters_df["cluster"].nunique()
print(f"Cluster IDs remapped to 0…{num_clusters_orig - 1}")

# Build dataset
data_df = pd.DataFrame({
    "Playlist Title": [playlist_titles.get(pid, "") for pid in clusters_df["pid"]],
    "Cluster Labels": clusters_df["cluster"],
})
data_df = data_df[data_df["Playlist Title"].str.strip() != ""].reset_index(drop=True)
print(f"Dataset after dropping untitled playlists: {len(data_df)}")

# 90 / 10 train–val split
train_df, val_df = train_test_split(data_df, test_size=0.1, random_state=42)

TRAIN_CSV = os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_train.csv")
VAL_CSV   = os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_val.csv")

train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV,   index=False)
print(f"Train: {len(train_df)}  Val: {len(val_df)}")
display(data_df.head())


In [ ]:
hard_ft.run(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    output_dir=FINETUNED_MODEL_DIR,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=128,
    epochs=20,
    learning_rate=3e-4,
    warmup_steps=200,
)


---
## Part 2: Pruned HDBSCAN Clusters

Fine-tune on the LSH-pruned cluster assignments (`hdbscan_clusters_pruned.csv`).
Overly broad clusters have already been removed by `lsh_cluster_picking.prune_clusters`.
Cluster IDs are remapped to 0…N-1 again since pruning makes them non-contiguous.


#### Load and prepare pruned HDBSCAN cluster dataset


In [ ]:
clusters_pruned_df = pd.read_csv(
    os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_clusters_pruned.csv"),
    dtype={"pid": str},
)
print(f"Loaded {len(clusters_pruned_df)} rows, {clusters_pruned_df['cluster'].nunique()} clusters (post-pruning)")

# Remap cluster IDs to contiguous 0…N-1
unique_ids_pruned  = sorted(clusters_pruned_df["cluster"].unique())
id_map_pruned      = {old: new for new, old in enumerate(unique_ids_pruned)}
clusters_pruned_df["cluster"] = clusters_pruned_df["cluster"].map(id_map_pruned)
num_clusters_pruned = clusters_pruned_df["cluster"].nunique()
print(f"Cluster IDs remapped to 0…{num_clusters_pruned - 1}")

# Build dataset
data_pruned_df = pd.DataFrame({
    "Playlist Title": [playlist_titles.get(pid, "") for pid in clusters_pruned_df["pid"]],
    "Cluster Labels": clusters_pruned_df["cluster"],
})
data_pruned_df = data_pruned_df[data_pruned_df["Playlist Title"].str.strip() != ""].reset_index(drop=True)
print(f"Dataset after dropping untitled playlists: {len(data_pruned_df)}")

# 90 / 10 train–val split
train_pruned_df, val_pruned_df = train_test_split(data_pruned_df, test_size=0.1, random_state=42)

TRAIN_PRUNED_CSV = os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_train_pruned.csv")
VAL_PRUNED_CSV   = os.path.join(CLUSTERS_OUTPUT_DIR, "hdbscan_val_pruned.csv")

train_pruned_df.to_csv(TRAIN_PRUNED_CSV, index=False)
val_pruned_df.to_csv(VAL_PRUNED_CSV,   index=False)
print(f"Train: {len(train_pruned_df)}  Val: {len(val_pruned_df)}")
display(data_pruned_df.head())


In [ ]:
hard_ft.run(
    train_csv=TRAIN_PRUNED_CSV,
    val_csv=VAL_PRUNED_CSV,
    output_dir=FINETUNED_PRUNED_MODEL_DIR,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    batch_size=128,
    epochs=20,
    learning_rate=3e-4,
    warmup_steps=200,
)
